# Day 5: Dimensionality Reduction
## Bioinformatics Course - IMBB-FORTH

### Objectives

- Understand what dimensionality reduction does and why it's useful
- Run PCA step by step and interpret the results
- Understand what UMAP does differently
- Compare PCA and UMAP on single-cell data

---

## 1. Setup

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

sns.set_theme(style="white")

REPO = "https://raw.githubusercontent.com/cgenomicslab/imbb-data-analysis/main"

print("Libraries loaded.")

---

## 2. The Problem: Too Many Variables

Imagine you have 6 items described by 10 properties. Each property ranges from 0 to 1.

Can you tell which items are similar just by looking at 10 numbers?

In [ ]:
# 6 items, 10 properties (values between 0 and 1)
toy = pd.DataFrame({
    'p1':  [1.0, 0.9, 0.8, 0.0, 0.1, 0.2],
    'p2':  [0.9, 1.0, 0.9, 0.1, 0.0, 0.1],
    'p3':  [0.8, 0.9, 1.0, 0.2, 0.1, 0.0],
    'p4':  [0.0, 0.1, 0.2, 1.0, 0.9, 0.8],
    'p5':  [0.1, 0.0, 0.1, 0.9, 1.0, 0.9],
    'p6':  [0.2, 0.1, 0.0, 0.8, 0.9, 1.0],
    'p7':  [1.0, 0.5, 0.0, 1.0, 0.5, 0.0],
    'p8':  [1.0, 0.5, 0.0, 1.0, 0.5, 0.0],
    'p9':  [0.0, 0.5, 1.0, 0.0, 0.5, 1.0],
    'p10': [0.0, 0.5, 1.0, 0.0, 0.5, 1.0],
}, index=['A', 'B', 'C', 'D', 'E', 'F'])

toy

There are **two hidden trends** in this table:

1. **Trend 1 (p1–p6):** A, B, C score high on p1–p3 and low on p4–p6. D, E, F are the opposite.
2. **Trend 2 (p7–p10):** A and D score high on p7–p8, C and F score high on p9–p10, while B and E sit in the middle.

But with 10 properties you can’t make a simple plot.
[**PCA** (Principal Component Analysis)](https://en.wikipedia.org/wiki/Principal_component_analysis) compresses these 10 dimensions into 2 that capture the main **trends** in the data.

### Run PCA on the toy data

In [ ]:
# Run PCA: reduce 10 properties to 2 dimensions
pca = PCA(n_components=2)
coords = pca.fit_transform(toy)

# Results
toy_pca = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1]
}, index=toy.index)

toy_pca

In [ ]:
# Plot — axes through origin
pc1_var = round(pca.explained_variance_ratio_[0] * 100, 1)
pc2_var = round(pca.explained_variance_ratio_[1] * 100, 1)

sns.scatterplot(data=toy_pca, x='PC1', y='PC2', s=200)

# Add item labels
for name, row in toy_pca.iterrows():
    plt.text(row['PC1'] + 0.05, row['PC2'] + 0.05, name, fontsize=12)

# Axes through origin
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
sns.despine()

plt.xlabel('PC1 (' + str(pc1_var) + '%)')
plt.ylabel('PC2 (' + str(pc2_var) + '%)')
plt.title('PCA of toy data')
plt.show()

print("PC1 (left vs right): A, B, C separate from D, E, F.")
print("PC2 (top vs bottom): A and D separate from C and F. B and E are neutral.")
print("PCA captured both trends using just 2 numbers per item instead of 10.")

### What did PCA do?

Each **principal component** is a **trend** in the data — a direction along which the items vary the most.

- **PC1** captures the **strongest trend** (~60%): the difference between {A, B, C} and {D, E, F} (driven by p1–p6).
- **PC2** captures the **second trend** (~39%): the difference between {A, D} at the top and {C, F} at the bottom (driven by p7–p10). B and E sit at zero on PC2 because they are neutral on this trend.

Together, these two trends account for nearly all the variation in the data.

The percentage next to each axis tells you how much of the total variation that trend explains.

In [ ]:
print("PC1 captures:", pc1_var, "% of the total variation")
print("PC2 captures:", pc2_var, "%")
print("Together:    ", round(pc1_var + pc2_var, 1), "%")
print()
print("We kept almost all the information while going from 10 dimensions to 2.")

---

## 3. PCA on Gene Expression Data

We have 100 genes measured across 9 samples from 3 conditions.
Each sample is a point in 100-dimensional space — we can't visualize that.
PCA will find the main trends and project the samples onto 2 dimensions.

### Load the data

In [ ]:
expression = pd.read_csv(REPO + '/data/bulk_rnaseq_counts.csv', index_col=0)
metadata = pd.read_csv(REPO + '/data/bulk_rnaseq_metadata.csv')

print("Expression:", expression.shape[0], "samples x", expression.shape[1], "genes")
print("Conditions:", list(metadata['condition'].unique()))
expression.head()

### Step 1: Log-transform

Raw counts are very skewed — a few highly expressed genes would dominate.
Log-transforming compresses the range and makes the data more symmetric.

In [ ]:
# Log-transform
log_expression = np.log2(expression + 1)

print("Before log: Gene_001 =", list(expression['Gene_001'].tolist()))
print("After log:  Gene_001 =", [round(v, 1) for v in log_expression['Gene_001'].tolist()])

### Step 2: Scale

Different genes have different expression ranges.
**Scaling** makes each gene have mean = 0 and standard deviation = 1,
so no single gene dominates the PCA.

In [ ]:
# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(log_expression)

print("Scaled data shape:", X_scaled.shape)
print("Ready for PCA.")

### Step 3: Run PCA

In [ ]:
# Run PCA
pca_expr = PCA(n_components=5)
coords = pca_expr.fit_transform(X_scaled)

# Results
expr_pca = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'condition': metadata['condition'].values,
    'sample': metadata['sample_name'].values
})

expr_pca

### Step 4: Plot

In [ ]:
# Variance explained
pc1_var = round(pca_expr.explained_variance_ratio_[0] * 100, 1)
pc2_var = round(pca_expr.explained_variance_ratio_[1] * 100, 1)

# PCA plot
sns.scatterplot(data=expr_pca, x='PC1', y='PC2', hue='condition', s=200)

plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
sns.despine()
plt.xlabel('PC1 (' + str(pc1_var) + '%)')
plt.ylabel('PC2 (' + str(pc2_var) + '%)')
plt.title('PCA of gene expression')
plt.show()

print("Samples from the same condition cluster together.")
print("PC1 (the strongest trend) separates the conditions.")

### Scree plot: how many PCs matter?

In [ ]:
# Which PCs capture the most variation?
pc_labels = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']
var_pct = [round(v * 100, 1) for v in pca_expr.explained_variance_ratio_]

sns.barplot(x=pc_labels, y=var_pct)
plt.ylabel('Variance explained (%)')
plt.title('Scree plot')
plt.show()

for i in range(len(pc_labels)):
    print(pc_labels[i] + ":", var_pct[i], "%")

### What drives PC1?

The **loadings** tell us which genes contribute most to each trend.

In [ ]:
# Top genes for PC1
pc1_loadings = pd.DataFrame({
    'gene': expression.columns,
    'loading': pca_expr.components_[0]
})
pc1_loadings['abs_loading'] = pc1_loadings['loading'].abs()

print("Top 10 genes driving PC1 (the main trend):")
pc1_loadings.sort_values('abs_loading', ascending=False).head(10)

### 💡 Exercise 5.1

1. How much total variance do PC1 and PC2 together explain?
2. Try plotting PC1 vs PC3 — does it reveal anything new?
3. What would happen if you skip the scaling step?

In [ ]:
# Your code here

# 1. Total variance

# 2. PC1 vs PC3

# 3. PCA without scaling


---

## 4. Single-Cell Data: PCA vs UMAP

With only 9 bulk samples, PCA works perfectly.
But what about datasets with **hundreds or thousands** of samples, like single-cell RNA-seq?

We have expression data for 200 individual cells from 4 immune cell types.
Let's compare PCA and [**UMAP**](https://umap-learn.readthedocs.io/) on this dataset.

### Load single-cell data

In [ ]:
# Load single-cell expression (200 cells x 100 genes)
sc_expression = pd.read_csv(REPO + '/data/singlecell_expression.csv', index_col=0)
sc_metadata = pd.read_csv(REPO + '/data/singlecell_metadata.csv')

print("Expression:", sc_expression.shape[0], "cells x", sc_expression.shape[1], "genes")
print("Metadata:", sc_metadata.shape[0], "cells")
print("Cell types:", dict(sc_metadata['cell_type'].value_counts()))

### PCA on single-cell data

In [ ]:
# Same steps: log-transform, scale, run PCA
sc_log = np.log2(sc_expression + 1)
sc_scaled = StandardScaler().fit_transform(sc_log)

sc_pca = PCA(n_components=2)
sc_pca_coords = sc_pca.fit_transform(sc_scaled)

sc_pca_df = pd.DataFrame({
    'PC1': sc_pca_coords[:, 0],
    'PC2': sc_pca_coords[:, 1],
    'cell_type': sc_metadata['cell_type'].values
})

pc1_var = round(sc_pca.explained_variance_ratio_[0] * 100, 1)
pc2_var = round(sc_pca.explained_variance_ratio_[1] * 100, 1)

sns.scatterplot(data=sc_pca_df, x='PC1', y='PC2', hue='cell_type', s=30, alpha=0.7)
plt.axhline(0, color='black', linewidth=0.8)
plt.axvline(0, color='black', linewidth=0.8)
sns.despine()
plt.xlabel('PC1 (' + str(pc1_var) + '%)')
plt.ylabel('PC2 (' + str(pc2_var) + '%)')
plt.title('PCA of single-cell data')
plt.show()

print("PCA separates some cell types, but the clusters overlap.")

### UMAP on single-cell data

[UMAP](https://umap-learn.readthedocs.io/) (Uniform Manifold Approximation and Projection) is a **non-linear** method that focuses on preserving **local neighborhoods**:
cells that are similar stay close together.

This makes it much better at finding **clusters** in complex data.

In [ ]:
from umap import UMAP

# Run UMAP on the same scaled data
umap_model = UMAP(n_components=2, random_state=42)
umap_coords = umap_model.fit_transform(sc_scaled)

sc_umap_df = pd.DataFrame({
    'UMAP1': umap_coords[:, 0],
    'UMAP2': umap_coords[:, 1],
    'cell_type': sc_metadata['cell_type'].values
})

with sns.axes_style("whitegrid"):
    sns.scatterplot(data=sc_umap_df, x='UMAP1', y='UMAP2', hue='cell_type', s=30, alpha=0.7)
    plt.title('UMAP of single-cell data')
    plt.show()

print("UMAP forms much tighter, more distinct clusters.")

### Side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA panel — axes through origin
sns.scatterplot(data=sc_pca_df, x='PC1', y='PC2', hue='cell_type', s=30, alpha=0.7, ax=axes[0])
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('PC1 (' + str(pc1_var) + '%)')
axes[0].set_ylabel('PC2 (' + str(pc2_var) + '%)')
axes[0].set_title('PCA')
for spine in ['top', 'right']:
    axes[0].spines[spine].set_visible(False)

# UMAP panel — normal grid
axes[1].grid(True, color='lightgray', linewidth=0.5)
sns.scatterplot(data=sc_umap_df, x='UMAP1', y='UMAP2', hue='cell_type', s=30, alpha=0.7, ax=axes[1])
axes[1].set_title('UMAP')

plt.tight_layout()
plt.show()

print("PCA captures global trends (variance explained).")
print("UMAP reveals local structure (tighter clusters).")
print("For single-cell data, UMAP is the standard visualization method.")

### When to use each?

| | PCA | UMAP |
|---|---|---|
| Best for | Understanding which trends dominate | Visualizing clusters |
| Type | Linear | Non-linear |
| Interpretable? | Yes (loadings, variance explained) | Less so |
| Typical use | Bulk RNA-seq, quality control | Single-cell RNA-seq |
| Links | [Wikipedia](https://en.wikipedia.org/wiki/Principal_component_analysis) | [UMAP docs](https://umap-learn.readthedocs.io/) |

### 💡 Exercise 5.2

**Task:**
1. In the PCA plot, which two cell types overlap the most? Why might that be?
2. Run UMAP with a different `n_neighbors` value (try 5 and 50). What changes?
3. Pick one cell type and find its marker genes (genes with highest mean expression in that type vs others).

In [ ]:
# Your code here

# 2. UMAP with different n_neighbors
# umap_5 = UMAP(n_components=2, n_neighbors=5, random_state=42)
# umap_50 = UMAP(n_components=2, n_neighbors=50, random_state=42)

# 3. Marker genes for a cell type
# Hint: compare mean expression of each gene between one type and the rest


---

## Summary

### Dimensionality reduction
- **The problem:** too many variables to visualize
- **The solution:** find the main **trends** and project the data onto 2 dimensions

### PCA
- Each PC is a **trend** — a direction of variation in the data
- **Variance explained** tells you how important each trend is
- **Loadings** tell you which variables drive each trend
- Always **log-transform** and **scale** expression data before PCA

### UMAP
- Preserves **local neighborhoods** — similar items stay close
- Better for visualizing **clusters** in complex data
- Standard for single-cell RNA-seq visualization

### Key Functions
- `StandardScaler().fit_transform(X)` — scale data
- `PCA(n_components=2).fit_transform(X)` — run PCA
- `UMAP(n_components=2).fit_transform(X)` — run UMAP

---

## Week 1 overview

1. **Day 1:** Python basics — variables, lists, loops, functions
2. **Day 2:** Data exploration — pandas, filtering, groupby, seaborn
3. **Day 3:** Statistics — p-values, permutation tests, t-test, correlation
4. **Day 4:** Enrichment — hypergeometric test, differential expression, GO terms
5. **Day 5:** Dimensionality reduction — PCA, UMAP